In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
df_customers = spark.read.csv('olist_customers_dataset.csv', header=True, inferSchema=True)
df_customers.show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [4]:
from pyspark.sql.functions import col, upper, regexp_replace

# Converter o tipo da coluna ‘customer_zip_code_prefix’ para String
df_customers = df_customers.withColumn('customer_zip_code_prefix', col('customer_zip_code_prefix').cast('string'))

# Colocar nomes de cidade em letra maiúscula
df_customers = df_customers.withColumn('customer_city', upper(col('customer_city')))

# Remover acentos
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[áàâãä]', 'a'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[éèêë]', 'e'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[íìîï]', 'i'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[óòôõö]', 'o'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), '[úùûü]', 'u'))
df_customers = df_customers.withColumn('customer_city', regexp_replace(col('customer_city'), 'ç', 'c'))


# Retirar todos estados que não sejam SP
df_customers_sp = df_customers.filter(col('customer_state') == 'SP')

# Remover a coluna customer_state
df_customers_sp = df_customers_sp.drop("customer_state")

df_customers_sp.show()

+--------------------+--------------------+------------------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|
+--------------------+--------------------+------------------------+--------------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              FRANCA|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|SAO BERNARDO DO C...|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           SAO PAULO|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     MOGI DAS CRUZES|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            CAMPINAS|
|fd826e7cf63160e53...|addec96d2e059c80c...|                    4534|           SAO PAULO|
|b2d1536598b73a9ab...|918dc87cd72cd9f6e...|                   18682|    LENCOIS PAULISTA|
|eabebad39a88bb6f5...|295c05e81917928d7...|                    5704|           SAO PAULO|
|206f3129c

In [5]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

df_customers_final_df = df_customers_sp.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"customers_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
df_customers_final_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\customers_final_20260331_232800.csv
